# Pydantic to structure Gemini outputs

In [3]:
from dotenv import load_dotenv
import os 
from google import genai

load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)

print(response.text)

Why do programmers prefer dark mode?

Because light attracts bugs!

***

And here's another classic:

There are 10 types of people in the world: those who understand binary, and those who don't.


In [6]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

ask_llm("Du är en göteborgare, ge mig ett skämt som är gött")


'Hallå där, goa gubbe! Klart du ska ha ett skämt som är gött, la! Här kommer ett, va:\n\n***\n\nDet va två göteborgare som gick nere vid Göta Älv.\nDen ena snubblar till och tappar sin smörgås i vattnet.\n\n"Åh, va gött!" utbrister han.\n\nDen andre stirrar på honom och frågar: "Va sa du? Har du tappat smörgåsen i älven och säger \'Åh, va gött\'?"\n\n"Ja", svarar den förste, "För nu ligger den ju i **Göta** älv!"\n\n***\n\nÄr det inte för gött, så säg! Ha det så gott, la!'

# Try to get data from llm

In [ ]:
response = ask_llm("""
   Du är en expert inom köp och sälj av bostäder.

    Uppgift:
    Returnera ENDAST giltig JSON (ingen text, inga kodblock, inga backticks, ingen markdown).
    Returnera en JSON-lista med exakt 5 objekt.
    Varje objekt ska ha följande fält (och exakt dessa nycklar):
    - "address": string
    - "city": string
    - "price_sek": integer (heltal i SEK, 1_000_000–8_000_000)
    - "monthly_fee": integer (heltal i SEK, 500–8000)
    - "area": integer (kvm, 10–200)

    Regler:
    - Endast en JSON-lista: [ { ... }, { ... }, ... ]
    - Använd dubbelfnuttar runt alla strängar.
    - Inga kommentarer, inga extra fält, inga efterslängar.

    Exempel på format (OBS: detta är bara format, hitta på egna värden):
    [
      {"address":"Storgatan 1","city":"Göteborg","price_sek":3500000,"monthly_fee":3900,"area":62},
      {"address":"...","city":"...","price_sek":...,"monthly_fee":...,"area":...},
      ...
    ]

                    
  """)

response

'[\n  {\n    "address": "Vasagatan 15",\n    "city": "Stockholm",\n    "price_sek": 5750000,\n    "monthly_fee": 3800,\n    "area": 85\n  },\n  {\n    "address": "Linnégatan 42",\n    "city": "Göteborg",\n    "price_sek": 3200000,\n    "monthly_fee": 4500,\n    "area": 60\n  },\n  {\n    "address": "Södra Förstadsgatan 12A",\n    "city": "Malmö",\n    "price_sek": 2100000,\n    "monthly_fee": 2900,\n    "area": 45\n  },\n  {\n    "address": "Storgatan 30",\n    "city": "Uppsala",\n    "price_sek": 4100000,\n    "monthly_fee": 5200,\n    "area": 75\n  },\n  {\n    "address": "Drottninggatan 5",\n    "city": "Örebro",\n    "price_sek": 1850000,\n    "monthly_fee": 1950,\n    "area": 30\n  }\n]'

In [29]:
print(response)

[
  {
    "address": "Vasagatan 15",
    "city": "Stockholm",
    "price_sek": 5750000,
    "monthly_fee": 3800,
    "area": 85
  },
  {
    "address": "Linnégatan 42",
    "city": "Göteborg",
    "price_sek": 3200000,
    "monthly_fee": 4500,
    "area": 60
  },
  {
    "address": "Södra Förstadsgatan 12A",
    "city": "Malmö",
    "price_sek": 2100000,
    "monthly_fee": 2900,
    "area": 45
  },
  {
    "address": "Storgatan 30",
    "city": "Uppsala",
    "price_sek": 4100000,
    "monthly_fee": 5200,
    "area": 75
  },
  {
    "address": "Drottninggatan 5",
    "city": "Örebro",
    "price_sek": 1850000,
    "monthly_fee": 1950,
    "area": 30
  }
]


## Parse and validate data

In [25]:
print(type(response))

<class 'str'>


In [30]:
from  pydantic import BaseModel
import json

class Apartment(BaseModel):
    address: str
    city: str
    price_sek: int
    monthly_fee: int
    area: int

class ApartmentList(BaseModel):
    objects: list[Apartment]

apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments

ApartmentList(objects=[Apartment(address='Vasagatan 15', city='Stockholm', price_sek=5750000, monthly_fee=3800, area=85), Apartment(address='Linnégatan 42', city='Göteborg', price_sek=3200000, monthly_fee=4500, area=60), Apartment(address='Södra Förstadsgatan 12A', city='Malmö', price_sek=2100000, monthly_fee=2900, area=45), Apartment(address='Storgatan 30', city='Uppsala', price_sek=4100000, monthly_fee=5200, area=75), Apartment(address='Drottninggatan 5', city='Örebro', price_sek=1850000, monthly_fee=1950, area=30)])

In [31]:
apartments.objects

[Apartment(address='Vasagatan 15', city='Stockholm', price_sek=5750000, monthly_fee=3800, area=85),
 Apartment(address='Linnégatan 42', city='Göteborg', price_sek=3200000, monthly_fee=4500, area=60),
 Apartment(address='Södra Förstadsgatan 12A', city='Malmö', price_sek=2100000, monthly_fee=2900, area=45),
 Apartment(address='Storgatan 30', city='Uppsala', price_sek=4100000, monthly_fee=5200, area=75),
 Apartment(address='Drottninggatan 5', city='Örebro', price_sek=1850000, monthly_fee=1950, area=30)]

In [32]:
apartments.objects[1].address, apartments.objects[1].city

('Linnégatan 42', 'Göteborg')

In [41]:
addresses = [apartment.address for apartment in apartments.objects ]
addresses



['Vasagatan 15',
 'Linnégatan 42',
 'Södra Förstadsgatan 12A',
 'Storgatan 30',
 'Drottninggatan 5']

In [43]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]

addresses

['Linnégatan 42', 'Södra Förstadsgatan 12A', 'Drottninggatan 5']

In [44]:
addresses_4_to_8 =[
    [apartments.address, apartments.city, apartments.price_sek, apartments.monthly_fee]
    for apartments
    in apartments.objects
    if apartments.price_sek > 4000000 and apartments.price_sek < 8000000
]

addresses_4_to_8

[['Vasagatan 15', 'Stockholm', 5750000, 3800],
 ['Storgatan 30', 'Uppsala', 4100000, 5200]]

In [45]:
import pandas as pd 
import duckdb

df_filtered_apartments = pd.DataFrame(
    addresses_4_to_8,
    columns=["address", "city", "price_sek", "monthly_fee"]
)

df_filtered_apartments

,address,city,price_sek,monthly_fee
0,Vasagatan 15,Stockholm,5750000,3800
1,Storgatan 30,Uppsala,4100000,5200


In [46]:
df_filtered_apartments.to_csv("filtered_homes.csv", index=False)

## save to JSON - serialize pydantic model

In [ ]:
#dictionary
apartments.model_dump()

{'objects': [{'address': 'Vasagatan 15',
   'city': 'Stockholm',
   'price_sek': 5750000,
   'monthly_fee': 3800,
   'area': 85},
  {'address': 'Linnégatan 42',
   'city': 'Göteborg',
   'price_sek': 3200000,
   'monthly_fee': 4500,
   'area': 60},
  {'address': 'Södra Förstadsgatan 12A',
   'city': 'Malmö',
   'price_sek': 2100000,
   'monthly_fee': 2900,
   'area': 45},
  {'address': 'Storgatan 30',
   'city': 'Uppsala',
   'price_sek': 4100000,
   'monthly_fee': 5200,
   'area': 75},
  {'address': 'Drottninggatan 5',
   'city': 'Örebro',
   'price_sek': 1850000,
   'monthly_fee': 1950,
   'area': 30}]}

In [49]:
#str of json data
apartments.model_dump_json()

'{"objects":[{"address":"Vasagatan 15","city":"Stockholm","price_sek":5750000,"monthly_fee":3800,"area":85},{"address":"Linnégatan 42","city":"Göteborg","price_sek":3200000,"monthly_fee":4500,"area":60},{"address":"Södra Förstadsgatan 12A","city":"Malmö","price_sek":2100000,"monthly_fee":2900,"area":45},{"address":"Storgatan 30","city":"Uppsala","price_sek":4100000,"monthly_fee":5200,"area":75},{"address":"Drottninggatan 5","city":"Örebro","price_sek":1850000,"monthly_fee":1950,"area":30}]}'

In [51]:
with open ("apartments.json", "w") as json_file:
    json_file.write(apartments.model_dump_json(indent=3))


## pandas dataframe other way

In [53]:
apartments.objects

[Apartment(address='Vasagatan 15', city='Stockholm', price_sek=5750000, monthly_fee=3800, area=85),
 Apartment(address='Linnégatan 42', city='Göteborg', price_sek=3200000, monthly_fee=4500, area=60),
 Apartment(address='Södra Förstadsgatan 12A', city='Malmö', price_sek=2100000, monthly_fee=2900, area=45),
 Apartment(address='Storgatan 30', city='Uppsala', price_sek=4100000, monthly_fee=5200, area=75),
 Apartment(address='Drottninggatan 5', city='Örebro', price_sek=1850000, monthly_fee=1950, area=30)]

In [67]:
addresses = [apartment.address for apartment in apartments.objects]
prices = [apartment.price_sek for apartment in apartments.objects]
monthly_fees = [apartment.monthly_fee for apartment in apartments.objects]
areas = [apartment.area for apartment in apartments.objects]

df = pd.DataFrame(
    {"address": addresses, "area": areas, "price": prices, "monthly_fee": monthly_fees}
)

df


,address,area,price,monthly_fee
0,Vasagatan 15,85,5750000,3800
1,Linnégatan 42,60,3200000,4500
2,Södra Förstadsgatan 12A,45,2100000,2900
3,Storgatan 30,75,4100000,5200
4,Drottninggatan 5,30,1850000,1950


In [69]:
df.to_dict()

{'address': {0: 'Vasagatan 15',
  1: 'Linnégatan 42',
  2: 'Södra Förstadsgatan 12A',
  3: 'Storgatan 30',
  4: 'Drottninggatan 5'},
 'area': {0: 85, 1: 60, 2: 45, 3: 75, 4: 30},
 'price': {0: 5750000, 1: 3200000, 2: 2100000, 3: 4100000, 4: 1850000},
 'monthly_fee': {0: 3800, 1: 4500, 2: 2900, 3: 5200, 4: 1950}}

In [71]:
import duckdb
import dlt

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    for record in df.to_dict(orient="records"):
        yield record 

pipeline = dlt.pipeline(
    pipeline_name="apartments", 
    destination="duckdb",
    dataset_name="staging"
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 0.26 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Skolprojekt\ai_engineering\ai_engineering_jonas_gustafsson\code-alongs\07b_pydantic_gemini\apartments.duckdb location to store data
Load package 1757334545.2409887 is LOADED and contains no failed jobs
